# Selecting OTel GenAI Semantic Convention Attributes in TruLens Evals

Short answer: **yes, for `gen_ai.*` span attributes.**

TruLens emits [OpenTelemetry GenAI semantic convention](https://opentelemetry.io/docs/specs/semconv/gen-ai/)
attributes alongside its own `ai.observability.*` attributes, and
`Selector.span_attribute` is an unvalidated raw attribute-name lookup. So you
can point any metric at a `gen_ai.*` key directly, without TruLens-specific
attribute knowledge.

This notebook demonstrates five selection styles against `gen_ai.*` attributes:

| Style | Selector | Example metric here |
| --- | --- | --- |
| Single attribute | `span_attribute="gen_ai.request.model"` | Model Allowlist |
| Per-item (fan out) | `span_attribute=...documents, collect_list=False` | Context Relevance |
| Whole list (fan in) | `span_attribute=...documents, collect_list=True` | Groundedness |
| Derived from several attributes | `span_attributes_processor=lambda attrs: ...` | Token Efficiency, Tool Argument Validity |
| Whole trace | `trace_level=True` | Logical Consistency, Single Model Per Trace |
| Whole conversation | `.on_conversation()` | Conversation Helpfulness |

For a worked coding-agent example — evaluating a Claude Code session assembled from client hooks — see [`coding_agent_trace_evaluation.ipynb`](./coding_agent_trace_evaluation.ipynb).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/truera/trulens/blob/main/examples/expositional/otel_genai/genai_semconv_attribute_selection.ipynb)

## What TruLens emits, and what it does not

`@instrument()` sets `gen_ai.*` attributes automatically based on `span_type`.
You do not call any GenAI-specific API — you set the TruLens attributes as
usual and the `gen_ai.*` mirror is written for you.

| `span_type` | `gen_ai.*` span attributes emitted |
| --- | --- |
| `GENERATION` | `gen_ai.operation.name`, `gen_ai.request.model`, `gen_ai.request.temperature`, `gen_ai.system`, `gen_ai.usage.input_tokens`, `gen_ai.usage.output_tokens` |
| `RETRIEVAL` | `gen_ai.retrieval.query.text`, `gen_ai.retrieval.documents` |
| `TOOL`, `MCP` | `gen_ai.tool.name`, `gen_ai.tool.call.arguments`, `gen_ai.tool.call.result` |

Message content is handled separately from metadata. Prompts and completions are
emitted as a `gen_ai.client.inference.operation.details` span *event* rather than
as span attributes, gated behind `TRULENS_OTEL_CAPTURE_CONTENT` for PII safety.
Every selector in this notebook reads span attributes, so it uses
`ai.observability.record_root.input` / `.output` — or the function-call
attributes — when it needs the prompt or the response.

One limit is worth knowing before you design around this: **spans from
third-party instrumentation are dropped.** TruLens' exporter keeps only spans
carrying `ai.observability.app_name`, so spans produced by e.g. `openinference`
or `opentelemetry-instrumentation-openai` are filtered out at export and never
become selectable. `gen_ai.*` selection works on spans TruLens itself produced.

`gen_ai.retrieval.query.text` and `gen_ai.retrieval.documents` are TruLens
aliases under the `gen_ai` namespace, not official OTel GenAI attributes — the
spec has no retrieval attributes yet.

## Setup

In [1]:
# !pip install trulens trulens-providers-cortex snowflake-snowpark-python

In [2]:
import json
import os

import pandas as pd
from snowflake.cortex import complete
from snowflake.snowpark import Session
from trulens.apps.app import TruApp
from trulens.core import Metric
from trulens.core import TruSession
from trulens.core.database.connector.default import DefaultDBConnector
from trulens.core.feedback.selector import Selector
from trulens.core.feedback.selector import Trace
from trulens.core.otel.instrument import instrument
from trulens.otel.semconv.trace import GenAIAttributes
from trulens.otel.semconv.trace import SpanAttributes
from trulens.providers.cortex import Cortex

Package jsonschema not present in requirements.


Connect to Snowflake. `APP_MODEL` generates answers; `JUDGE_MODEL` runs the
LLM-based metrics.

In [3]:
snowpark_session = Session.builder.config(
    "connection_name", os.environ.get("SNOWFLAKE_CONNECTION_NAME", "default")
).create()

APP_MODEL = "llama3.3-70b"
JUDGE_MODEL = "claude-sonnet-4-5"

provider = Cortex(snowpark_session=snowpark_session, model_engine=JUDGE_MODEL)

In [4]:
session = TruSession(
    connector=DefaultDBConnector(database_url="sqlite:///genai_semconv_demo.sqlite")
)
session.reset_database()

## An instrumented support agent

Four instrumented methods, one span type each. Note that nothing here mentions
`gen_ai` — the attributes below are the ordinary TruLens ones, and the
`gen_ai.*` mirror is derived from them.

The `GENERATION` span reads its `gen_ai.*` values from the TruLens cost
attributes (`COST.MODEL`, `COST.NUM_PROMPT_TOKENS`, `COST.NUM_COMPLETION_TOKENS`)
plus three plain keys that the instrumentor looks for by name:
`temperature`, `provider_name`, and `operation_name`. The `TOOL` span reads
`call_arguments` and `call_result` the same way.

In [5]:
KB = [
    "Password resets are self-serve from Account -> Security -> Reset password.",
    "A reset link stays valid for 30 minutes, then must be requested again.",
    "Resetting a password signs the account out of every active session.",
    "Refunds for annual plans are prorated from the cancellation date.",
    "Our headquarters is in Bellevue, Washington.",
]

ORDERS = {"A-1099": "shipped", "A-1100": "processing"}


def count_tokens(text: str) -> int:
    """Real token count from Snowflake Cortex."""
    return int(
        snowpark_session.sql(
            "select snowflake.cortex.count_tokens(?, ?)",
            params=[APP_MODEL, text or ""],
        ).collect()[0][0]
    )


class SupportAgent:
    @instrument(
        span_type=SpanAttributes.SpanType.RETRIEVAL,
        attributes={
            SpanAttributes.RETRIEVAL.QUERY_TEXT: "query",
            SpanAttributes.RETRIEVAL.RETRIEVED_CONTEXTS: "return",
        },
    )
    def retrieve(self, query: str) -> list:
        """Keyword overlap stand-in for a vector search."""
        terms = {t for t in query.lower().split() if len(t) > 3}
        scored = [
            (len(terms & set(doc.lower().split())), doc) for doc in KB
        ]
        scored.sort(key=lambda pair: pair[0], reverse=True)
        return [doc for _, doc in scored[:3]]

    @instrument(
        span_type=SpanAttributes.SpanType.TOOL,
        attributes=lambda ret, exception, *args, **kwargs: {
            "call_arguments": json.dumps({"order_id": kwargs.get("order_id")}),
            "call_result": json.dumps(ret),
        },
    )
    def lookup_order(self, order_id: str) -> dict:
        return {"order_id": order_id, "status": ORDERS.get(order_id, "unknown")}

    @instrument(
        span_type=SpanAttributes.SpanType.GENERATION,
        attributes=lambda ret, exception, *args, **kwargs: {
            SpanAttributes.COST.MODEL: APP_MODEL,
            SpanAttributes.COST.NUM_PROMPT_TOKENS: kwargs["prompt_tokens"],
            SpanAttributes.COST.NUM_COMPLETION_TOKENS: count_tokens(ret),
            # Plain keys the instrumentor maps into gen_ai.* by name.
            "temperature": 0.0,
            "provider_name": "snowflake.cortex",
            "operation_name": "chat",
        },
    )
    def generate(self, query: str, contexts: list, prompt_tokens: int) -> str:
        prompt = (
            "Answer the customer's question using only the context below. "
            "Be concise.\n\nContext:\n"
            + "\n".join(f"- {c}" for c in contexts)
            + f"\n\nQuestion: {query}"
        )
        return complete(
            APP_MODEL, [{"role": "user", "content": prompt}], session=snowpark_session
        ).strip()

    @instrument(
        span_type=SpanAttributes.SpanType.RECORD_ROOT,
        attributes={
            SpanAttributes.RECORD_ROOT.INPUT: "query",
            SpanAttributes.RECORD_ROOT.OUTPUT: "return",
        },
    )
    def answer(self, query: str) -> str:
        contexts = self.retrieve(query)
        for order_id in ORDERS:
            if order_id in query:
                contexts = contexts + [
                    f"Order {order_id} status: {self.lookup_order(order_id=order_id)['status']}."
                ]
        return self.generate(
            query=query,
            contexts=contexts,
            prompt_tokens=count_tokens(query + " ".join(contexts)),
        )

## Metrics

### 1. Single `gen_ai.*` attribute

The simplest case: name the attribute as a string. Nothing validates it against
`SpanAttributes`, which is exactly why arbitrary OTel keys work.

In [6]:
ALLOWED_MODELS = {"llama3.3-70b", "llama3.1-70b"}


def model_is_allowed(model: str) -> float:
    """Governance check: was the answer served by an approved model?"""
    return 1.0 if model in ALLOWED_MODELS else 0.0


m_model_allowlist = Metric(
    implementation=model_is_allowed,
    name="Model Allowlist",
).on({
    "model": Selector(
        span_type=SpanAttributes.SpanType.GENERATION,
        span_attribute=GenAIAttributes.REQUEST.MODEL,  # "gen_ai.request.model"
    )
})

### 2. Per-item selection — Context Relevance

`collect_list=False` fans the metric out over each element of
`gen_ai.retrieval.documents`, scoring one document per call, then averages. This
is the individual-attribute-selection pattern: the query comes from one
`gen_ai.*` attribute and each document from another, both on the same
`RETRIEVAL` span.

In [7]:
m_context_relevance = Metric(
    implementation=provider.context_relevance_with_cot_reasons,
    name="Context Relevance",
).on({
    "question": Selector(
        span_type=SpanAttributes.SpanType.RETRIEVAL,
        span_attribute=GenAIAttributes.RETRIEVAL.QUERY_TEXT,
    ),
    "context": Selector(
        span_type=SpanAttributes.SpanType.RETRIEVAL,
        span_attribute=GenAIAttributes.RETRIEVAL.DOCUMENTS,
        collect_list=False,  # one LLM call per retrieved document
    ),
})

### 3. Whole-list selection — Groundedness

Same attribute, `collect_list=True`: all documents arrive as one list so the
judge can check the answer against the full evidence set at once.

In [8]:
m_groundedness = Metric(
    implementation=provider.groundedness_measure_with_cot_reasons,
    name="Groundedness",
).on({
    "source": Selector(
        span_type=SpanAttributes.SpanType.RETRIEVAL,
        span_attribute=GenAIAttributes.RETRIEVAL.DOCUMENTS,
        collect_list=True,  # one LLM call against all documents
    ),
    "statement": Selector.select_record_output(),
})

### 4. Deriving a value from several `gen_ai.*` attributes

`span_attributes_processor` receives the span's whole attribute dict, so you can
combine keys, reshape them, or return `None` to skip the span (with
`ignore_none_values=True`). Here two `gen_ai.usage.*` counters become one
efficiency score.

In [9]:
def token_efficiency(usage: dict) -> float:
    """Penalise answers that spend a large prompt for a small completion."""
    total = usage["input_tokens"] + usage["output_tokens"]
    if total == 0:
        return 0.0
    return min(usage["output_tokens"] / total * 4, 1.0)


m_token_efficiency = Metric(
    implementation=token_efficiency,
    name="Token Efficiency",
).on({
    "usage": Selector(
        span_type=SpanAttributes.SpanType.GENERATION,
        span_attributes_processor=lambda attrs: {
            "input_tokens": attrs.get(GenAIAttributes.USAGE.INPUT_TOKENS, 0),
            "output_tokens": attrs.get(GenAIAttributes.USAGE.OUTPUT_TOKENS, 0),
        },
    )
})

### 5. Tool spans

`gen_ai.tool.*` is selected the same way. This metric checks that the tool was
called with a well-formed order id and returned a known status.

In [10]:
def tool_call_is_valid(call: dict) -> float:
    order_id = json.loads(call["arguments"]).get("order_id", "")
    status = json.loads(call["result"]).get("status")
    well_formed = order_id.startswith("A-") and len(order_id) == 6
    return 1.0 if well_formed and status != "unknown" else 0.0


m_tool_validity = Metric(
    implementation=tool_call_is_valid,
    name="Tool Call Validity",
).on({
    "call": Selector(
        span_type=SpanAttributes.SpanType.TOOL,
        span_attributes_processor=lambda attrs: {
            "name": attrs.get(GenAIAttributes.TOOL.NAME),
            "arguments": attrs.get(GenAIAttributes.TOOL.CALL_ARGUMENTS, "{}"),
            "result": attrs.get(GenAIAttributes.TOOL.CALL_RESULT, "{}"),
        },
    )
})

### 6. Trace-level metrics

`Selector(trace_level=True)` hands the metric a `Trace` object covering every
span in the record instead of one span's attribute. It must be the only selector
on the metric.

Two flavours below. First a built-in agentic judge, which serialises the trace
for the LLM.

In [11]:
m_logical_consistency = Metric(
    implementation=provider.logical_consistency_with_cot_reasons,
    name="Logical Consistency",
).on({"trace": Selector(trace_level=True)})

Second, a deterministic trace-level check written against `gen_ai.*` directly.
`Trace.events` is a DataFrame of the trace's spans, so you can walk every span's
`record_attributes` and assert cross-span invariants — here, that a single trace
did not silently fan out across multiple models.

In [12]:
def single_model_per_trace(trace: Trace) -> float:
    models = set()
    for _, row in trace.events.iterrows():
        attrs = row["record_attributes"]
        if not isinstance(attrs, dict):
            attrs = json.loads(attrs)
        model = attrs.get(GenAIAttributes.REQUEST.MODEL)
        if model:
            models.add(model)
    return 1.0 if len(models) <= 1 else 0.0


m_single_model = Metric(
    implementation=single_model_per_trace,
    name="Single Model Per Trace",
).on({"trace": Selector(trace_level=True)})

### 7. Conversation-level metrics

`.on_conversation()` groups every `RECORD_ROOT` sharing a `conversation_id`,
orders them by start time, and passes the ordered turns as
`[{"input": ..., "output": ...}, ...]`. The score is attached to the last record
of the conversation, so earlier turns show no value for it.

In [13]:
m_conversation_helpfulness = Metric(
    implementation=provider.conversation_helpfulness_with_cot_reasons,
    name="Conversation Helpfulness",
).on_conversation()

## Record a conversation

All metrics are attached at construction. Passing `conversation_id` to the
recording context is what makes the conversation-level metric possible.

In [14]:
agent = SupportAgent()
recorder = TruApp(
    agent,
    app_name="GenAI Semconv Selection",
    app_version=APP_MODEL,
    main_method=agent.answer,
    feedbacks=[
        m_model_allowlist,
        m_context_relevance,
        m_groundedness,
        m_token_efficiency,
        m_tool_validity,
        m_logical_consistency,
        m_single_model,
        m_conversation_helpfulness,
    ],
)

turns = [
    "How do I reset my password?",
    "How long does that reset link stay valid?",
    "What is the status of order A-1099?",
]

with recorder(conversation_id="support-thread-1") as recording:
    for turn in turns:
        print(f"Q: {turn}")
        print(f"A: {agent.answer(turn)}\n")

session.force_flush()

Q: How do I reset my password?


A: To reset your password, go to Account -> Security -> Reset password. This is a self-serve process.

Q: How long does that reset link stay valid?


A: 30 minutes.

Q: What is the status of order A-1099?


A: Order A-1099 has been shipped.



True

## Confirm the `gen_ai.*` attributes actually landed

Before trusting any selector, look at what is on the spans. This is also the
fastest way to debug an empty metric column.

In [15]:
events = session.get_events(
    app_name="GenAI Semconv Selection", app_version=APP_MODEL
)

rows = []
for _, event in events.iterrows():
    attrs = event["record_attributes"]
    if not isinstance(attrs, dict):
        attrs = json.loads(attrs)
    span_type = attrs.get(SpanAttributes.SPAN_TYPE)
    for key, value in attrs.items():
        if key.startswith("gen_ai"):
            text = str(value)
            rows.append({
                "span_type": span_type,
                "gen_ai attribute": key,
                "value": text[:70] + ("..." if len(text) > 70 else ""),
            })

pd.DataFrame(rows).drop_duplicates(
    subset=["span_type", "gen_ai attribute"]
).sort_values(["span_type", "gen_ai attribute"]).reset_index(drop=True)

,span_type,gen_ai attribute,value
0,generation,gen_ai.operation.name,chat
1,generation,gen_ai.request.model,llama3.3-70b
2,generation,gen_ai.request.temperature,0.0
3,generation,gen_ai.system,snowflake.cortex
4,generation,gen_ai.usage.input_tokens,51
5,generation,gen_ai.usage.output_tokens,23
6,retrieval,gen_ai.retrieval.documents,['Password resets are self-serve from Account ...
7,retrieval,gen_ai.retrieval.query.text,How do I reset my password?
8,tool,gen_ai.tool.call.arguments,"{""order_id"": ""A-1099""}"
9,tool,gen_ai.tool.call.result,"{""order_id"": ""A-1099"", ""status"": ""shipped""}"


## Compute and inspect the metrics

In [16]:
recorder.compute_feedbacks(raise_error_on_no_feedbacks_computed=False)
session.force_flush()

/Users/jreini/Desktop/development/git-sfc/trulens/src/feedback/trulens/feedback/llm_provider.py:3407: UserWarning: Failed to process and remove trivial statements. Proceeding with all statements.
  hypotheses = self._remove_trivial_statements(hypotheses)


True

In [17]:
records, metric_names = session.get_records_and_feedback(
    app_name="GenAI Semconv Selection"
)

present = [name for name in metric_names if name in records.columns]
records[["input"] + present].round(3)

,input,Model Allowlist,Context Relevance,Groundedness,Token Efficiency,Logical Consistency,Single Model Per Trace,Conversation Helpfulness,Tool Call Validity
0,How do I reset my password?,1.0,0.556,1.000,1.000,0.333,1.0,NaN,NaN
1,How long does that reset link stay valid?,1.0,0.556,0.667,0.281,0.333,1.0,NaN,NaN
2,What is the status of order A-1099?,1.0,0.000,0.000,0.541,0.000,1.0,0.667,1.0


Two things in this table are worth reading carefully, because both are selector
behaviour rather than defects.

`Conversation Helpfulness` is populated only on the final turn. Conversation
metrics attach their score to the last record in the conversation by design.

Turn 3 (`What is the status of order A-1099?`) scores 0.0 on both
`Context Relevance` and `Groundedness`, while `Tool Call Validity` scores 1.0.
That is correct: the answer came from the `lookup_order` tool, but both of those
metrics select their evidence from `gen_ai.retrieval.documents` on the
`RETRIEVAL` span, which holds knowledge-base passages only. The metrics are
telling you the answer is not supported *by retrieval* — which it is not. It is a
concrete demonstration that a selector defines the evidence scope, and that
scope is a modelling decision. If you want tool output judged as evidence too,
select it — for example with a `span_attributes_processor` over
`gen_ai.tool.call.result`, or by widening to `trace_level=True`.

The agentic `Logical Consistency` scores are low for the same class of reason:
its rubric rewards multi-step reasoning, and these traces are a single
retrieve-then-generate hop with little reasoning to assess.

## Dashboard

In [ ]:
from trulens.dashboard import run_dashboard

run_dashboard(session)

## Takeaways

- `Selector(span_attribute=...)` is a raw attribute lookup, so any `gen_ai.*`
  key TruLens persisted is selectable — no mapping layer required.
- `collect_list` decides fan-out vs. fan-in over a list-valued attribute; that
  is the whole difference between per-context relevance and whole-context
  groundedness.
- `span_attributes_processor` covers everything else: combining attributes,
  reshaping JSON, or filtering spans by value (return `None` and set
  `ignore_none_values=True`).
- `trace_level=True` and `.on_conversation()` widen the scope from one span to a
  whole trace or a whole thread, and both compose with `gen_ai.*` reads.
- Message content is carried on span events rather than span attributes, so the
  selectors here read `record_root.input` / `.output` for prompts and responses.
- Foreign OTel spans are dropped at export, so `gen_ai.*` selection applies to
  TruLens-produced spans only.